# 엔티티 추출 비교: 언어와 GLiNER 체크포인트

[Hybrid와 LLM-only 비교](01_entity_extraction_hybrid_vs_llm.ipynb)에서 확인한 GLiNER 오류가 한국어 입력에 고유한 문제인지, 체크포인트를 바꾸면 해소되는지를 확인한다.
노트북 01의 한국어 메시지 4개와 이를 직접 번역한 영어 문장 4개를 다음 다섯 가지 방식으로 추출한다.

| 모드 | 구성 |
|---|---|
| `gliner_base` | GLiNER `fastino/gliner2.5-base-v1` 단독 (현재 설정의 체크포인트) |
| `gliner_multi` | GLiNER `fastino/gliner2.5-multi-v1` 단독 (다국어 체크포인트) |
| `llm_only` | LLM 단독 |
| `hybrid_base` | 같은 실행의 `gliner_base` + `llm_only` 결과를 SDK 병합 로직으로 결합 |
| `hybrid_multi` | 같은 실행의 `gliner_multi` + `llm_only` 결과를 SDK 병합 로직으로 결합 |

Hybrid 결과는 LLM을 다시 호출하지 않고 동일한 LLM 응답에 GLiNER 결과만 더하여 만든다. 따라서 Hybrid와 LLM-only의 차이는 전적으로 GLiNER 추가의 효과이다. Neo4j에는 연결하지 않는다.

## 결과 요약

> 아래 수치는 2026-11-09 실행 결과(`entity_extraction_languages.json`, 문장당 3회 반복)이다.

| 모드 | 한국어 메시지 1 (GCP) | 한국어 메시지 4 (오라클) | 영어 | 지연 시간 중앙값 |
|---|---|---|---|---|
| `gliner_base` | `GCP에서` ✗, Cloud DNS 누락 | **PERSON ✗** + `IP 기능을` | Oracle ORG ✓, 메시지 1 엔티티 없음 | 146 ms |
| `gliner_multi` | `GCP에서` ✗, Cloud DNS 누락 | ORGANIZATION ✓ | `recurring(EVENT)` 노이즈 | 148 ms |
| `llm_only` | `GCP` ✓, `Cloud DNS` ✓ | ORGANIZATION ✓ | 정상, 관계 15개 | 1,404 ms |
| `hybrid_base` | 정답 + `GCP에서` | ORGANIZATION + **PERSON** | `service(ORGANIZATION)` 추가 | – |
| `hybrid_multi` | 정답 + `GCP에서` | ORGANIZATION ✓ | `service(ORGANIZATION)` 추가 | – |

- **조사 문제는 두 체크포인트 공통이다.** `GCP 에서`로 띄어 쓰면 둘 다 `GCP(ORGANIZATION)`을 정확히 추출한다.
- **오라클 타입 오류는 문자 체계와 문맥의 조합에 따라 달라진다.** base는 한국어 문맥의 `오라클`을, multi는 영어 문맥의 `오라클`을 PERSON으로 분류한다. 어느 체크포인트도 일관되지 않다.
- multi는 한국어 메시지 4의 PERSON 오류를 없애지만 `11월(EVENT)` 같은 새로운 노이즈를 만든다.
- GLiNER는 LLM보다 약 10배 빠르지만, Hybrid 구성에서는 어차피 LLM을 호출하므로 속도 이점이 남지 않는다.

## 실험 조건

| 항목 | 값 |
|---|---|
| SDK | `neo4j-agent-memory` 0.8.0, `gliner2` 2.1.0, `torch` 2.15.0, `transformers` 5.22.1 |
| 체크포인트 리비전 | base `07d99c0b`, multi `7b005274` |
| 실행 환경 | Darwin arm64, GLiNER CPU 실행, torch 8 threads |
| 온톨로지·threshold | POLE+O, `gliner_threshold` 0.2, `confidence_threshold` 0.0 (모든 모드 공통) |
| 반복 | 문장 쌍 4개 × 2개 언어 × 3회, 반복마다 언어·모드 실행 순서를 교대 |
| 지연 시간 | 명시적 워밍업 이후 측정, 저장·임베딩·해소 제외, LLM은 네트워크 지연 포함 |

| # | 한국어 (노트북 01 원문) | 영어 (직접 번역) |
|---|---|---|
| 1 | 이제 GCP에서 매달 나가는 비용은 Cloud DNS 영역 3개, 월 약 $0.6뿐입니다. | The only recurring GCP cost now is for three Cloud DNS zones, about $0.60 per month. |
| 2 | 화면과 상관없는 서버 쪽 작업(공지 저장, 작업실 정보)은 이미 같이 진행하고 있어요. | The server-side tasks unrelated to the screens (saving announcements and studio information) are already being worked on in parallel. |
| 3 | 11월 사용량을 서비스별로 조회해 보니 금액이 붙은 항목이 하나도 없습니다. | I checked November usage by service, and none of the items had any charges. |
| 4 | 오라클 공식 입장: 고정 IP 기능을 처음 발표할 때 'There is no charge to use this feature'라고 밝혔습니다. | Oracle's official position: when it first announced the static IP feature, it stated, 'There is no charge to use this feature'. |

## 준비

### 로그 출력 설정

In [ ]:
import logging

logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)
logging.getLogger("transformers.configuration_utils").setLevel(logging.ERROR)

### 프로젝트 경로와 메모리 설정

현재 작업 디렉터리에서 상위로 올라가며 `pyproject.toml`이 있는 프로젝트 루트를 찾아 모듈 검색 경로에 추가한다.

In [ ]:
from pathlib import Path
import sys

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv

from config.memory_client import load_memory_settings

load_dotenv(PROJECT / ".env")
settings = load_memory_settings()
RESULT_PATH = PROJECT / "experiments/entity_extraction_languages.json"
print(f"✓ Project root: {PROJECT}")
print(f"✓ Result file: {RESULT_PATH.relative_to(PROJECT)}")

### 실행 방식 선택

`RUN_EXPERIMENT = False`이면 저장된 결과 파일을 불러와 분석만 수행한다.
`True`로 바꾸면 두 GLiNER 체크포인트를 내려받아 로딩하고 LLM API를 호출하여 결과 파일을 덮어쓴다.

In [ ]:
RUN_EXPERIMENT = False
REPEATS = 3

### 보조 객체와 입력 문장

[Hybrid와 LLM-only 비교](01_entity_extraction_hybrid_vs_llm.ipynb)와 같은 보조 객체를 사용한다. 한국어 문장은 노트북 01에서 읽고, 영어 문장은 의미를 보존하여 직접 번역한 값을 사용한다.

In [ ]:
import ast
import json


class EmptyGraph:
    """No stored candidates; resolution runs only within each input message."""

    async def execute_read(self, query, parameters=None):
        return []

    async def execute_write(self, *args, **kwargs):
        raise RuntimeError("Database writes are disabled in this experiment")


def notebook_messages():
    """notebooks/01_short_term_memory.ipynb의 `messages` 리터럴을 읽는다."""
    notebook = json.loads((PROJECT / "notebooks/01_short_term_memory.ipynb").read_text())
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        tree = ast.parse("".join(cell["source"]))
        for node in tree.body:
            if isinstance(node, ast.Assign) and any(
                isinstance(target, ast.Name) and target.id == "messages"
                for target in node.targets
            ):
                return ast.literal_eval(node.value)
    raise ValueError("No literal messages list found in notebook 01")


def save(path, report):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")

In [ ]:
ENGLISH = [
    "The only recurring GCP cost now is for three Cloud DNS zones, about $0.60 per month.",
    "The server-side tasks unrelated to the screens (saving announcements and studio information) are already being worked on in parallel.",
    "I checked November usage by service, and none of the items had any charges.",
    "Oracle's official position: when it first announced the static IP feature, it stated, 'There is no charge to use this feature'.",
]
MODELS = {
    "gliner_base": "fastino/gliner2.5-base-v1",
    "gliner_multi": "fastino/gliner2.5-multi-v1",
}

## 실험

### 추출기 구성

GLiNER 두 모드는 `extractor_type=GLINER`에 체크포인트만 바꾸고, LLM 모드는 `extractor_type=LLM`으로 설정한다. 나머지 설정은 현재 프로젝트 설정을 그대로 사용한다.

In [ ]:
from neo4j_agent_memory import ExtractorType
from neo4j_agent_memory.extraction.factory import create_extractor, resolve_ontology
from neo4j_agent_memory.memory.short_term import ShortTermMemory


def build_components():
    ontology = resolve_ontology(settings.extraction)
    configs = {
        mode: settings.extraction.model_copy(update={
            "extractor_type": ExtractorType.GLINER, "gliner_model": model,
        }) for mode, model in MODELS.items()
    }
    configs["llm_only"] = settings.extraction.model_copy(update={"extractor_type": ExtractorType.LLM})
    extractors = {
        mode: create_extractor(config, settings.schema_config, settings.llm, ontology=ontology)
        for mode, config in configs.items()
    }
    storage = ShortTermMemory(EmptyGraph(), ontology=ontology,
                              validation_mode=settings.schema_config.validation_mode)

    def prepare(result):
        return storage._apply_ontology(result.filter_invalid_entities(ontology))

    return ontology, configs, extractors, prepare

### 반복 추출과 Hybrid 결합

1. 모든 추출기를 짧은 영어 문장으로 한 번씩 워밍업하여 모델 로딩 시간을 측정에서 제외한다.
2. 각 문장 쌍을 언어별로 세 추출기에 입력하고 실행 시간을 기록한다.
3. 같은 실행의 GLiNER 결과와 LLM 결과를 파이프라인과 동일한 순서(`merge_extraction_results` → `_apply_min_confidence` → `validate_relations`)로 결합하여 Hybrid 결과를 만든다.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import platform
from time import perf_counter

from neo4j_agent_memory.extraction.pipeline import (
    MergeStrategy, _apply_min_confidence, merge_extraction_results,
)


async def run_extraction(repeats, ontology, configs, extractors, prepare):
    import torch
    from huggingface_hub import HfApi

    korean = notebook_messages()
    assert len(korean) == len(ENGLISH) == 4
    report = {
        "status": "running", "started_at": datetime.now(timezone.utc).isoformat(),
        "repeats": repeats, "database_connected": False,
        "versions": {p: version(p) for p in ("neo4j-agent-memory", "gliner2", "torch", "transformers")},
        "hardware": {"system": platform.system(), "architecture": platform.machine(),
                     "device": settings.extraction.gliner_device, "torch_threads": torch.get_num_threads()},
        "model_revisions": {model: HfApi().model_info(model).sha for model in MODELS.values()},
        "configs": {mode: config.model_dump(mode="json") for mode, config in configs.items()},
        "message_pairs": [{"index": i, "ko": ko, "en": en}
                          for i, (ko, en) in enumerate(zip(korean, ENGLISH, strict=True), 1)],
        "warmup": [], "runs": [], "controlled_probes": [],
        "notes": [
            "English texts are manual translations of notebook 01; four pairs are a diagnostic sample, not a general language benchmark.",
            "All extractors share the current POLE+O ontology and threshold; GLiNER is on CPU.",
            "Hybrid outputs reuse each trial's LLM and GLiNER responses with SDK CONFIDENCE merging and storage filters.",
            "Latency excludes explicit warmup, storage, embeddings, and entity resolution; LLM time includes network latency.",
            "Controlled spelling/spacing probes are diagnostic edits, not proposed production preprocessing.",
        ],
    }
    save(RESULT_PATH, report)
    for mode, extractor in extractors.items():
        started = perf_counter()
        await extractor.extract("Alice works at Acme in Seoul.")
        report["warmup"].append({"mode": mode, "elapsed_ms": (perf_counter() - started) * 1000})
        save(RESULT_PATH, report)

    for repeat in range(1, repeats + 1):
        languages = ("ko", "en") if repeat % 2 else ("en", "ko")
        for pair in report["message_pairs"]:
            for language in languages:
                results = {}
                modes = list(extractors) if repeat % 2 else list(reversed(extractors))
                for mode in modes:
                    started = perf_counter()
                    raw = await extractors[mode].extract(pair[language])
                    elapsed_ms = (perf_counter() - started) * 1000
                    results[mode] = raw
                    prepared = prepare(raw)
                    report["runs"].append({
                        "repeat": repeat, "language": language, "message_index": pair["index"],
                        "mode": mode, "elapsed_ms": round(elapsed_ms, 3),
                        "raw_result": raw.model_dump(mode="json"),
                        "prepared_result": prepared.model_dump(mode="json"),
                    })
                    save(RESULT_PATH, report)
                    entities = ", ".join(f"{e.name}({e.type})" for e in prepared.entities) or "(none)"
                    print(f"{repeat}/{repeats} {language} message={pair['index']} {mode} "
                          f"{elapsed_ms:.0f}ms: {entities}")
                for gliner_mode in MODELS:
                    # Exactly reproduce the pipeline's post-stage merge steps.
                    merged = merge_extraction_results(
                        [results[gliner_mode], results["llm_only"]], MergeStrategy.CONFIDENCE,
                    )
                    merged = _apply_min_confidence(merged, settings.extraction.confidence_threshold)
                    merged, _ = merged.validate_relations(ontology, mode="drop")
                    report["runs"].append({
                        "repeat": repeat, "language": language, "message_index": pair["index"],
                        "mode": gliner_mode.replace("gliner_", "hybrid_"),
                        "paired_components": [gliner_mode, "llm_only"],
                        "prepared_result": prepare(merged).model_dump(mode="json"),
                    })
                save(RESULT_PATH, report)
    return report

### 통제 프로브

오류의 원인을 분리하기 위해 문장 하나만 바꾼 입력을 GLiNER 두 체크포인트에 넣는다. 이 편집은 원인 진단용이며, 운영 전처리 방안으로 제안하는 것이 아니다.

| 프로브 | 변경 | 확인하려는 것 |
|---|---|---|
| `ko_context_latin_oracle` | 한국어 메시지 4의 `오라클` → `Oracle` | 한국어 문맥에서 표기만 바꾸면 타입이 바뀌는가 |
| `en_context_hangul_oracle` | 영어 메시지 4의 `Oracle` → `오라클` | 영어 문맥에서 한글 표기를 쓰면 타입이 바뀌는가 |
| `ko_gcp_particle_separated` | 한국어 메시지 1의 `GCP에서` → `GCP 에서` | 조사를 띄어 쓰면 엔티티 범위가 정확해지는가 |

In [ ]:
async def run_probes(report, extractors, prepare):
    korean = [pair["ko"] for pair in report["message_pairs"]]
    probes = [
        ("ko_context_latin_oracle", korean[3].replace("오라클", "Oracle")),
        ("en_context_hangul_oracle", ENGLISH[3].replace("Oracle", "오라클")),
        ("ko_gcp_particle_separated", korean[0].replace("GCP에서", "GCP 에서")),
    ]
    for label, text in probes:
        for mode in MODELS:
            prepared = prepare(await extractors[mode].extract(text))
            report["controlled_probes"].append({"probe": label, "text": text, "mode": mode,
                "prepared_result": prepared.model_dump(mode="json")})
            print(f"PROBE {label} {mode}: {[(e.name, e.type) for e in prepared.entities]}")

### 관찰 지표 집계

언어·모드별로 주요 엔티티의 등장 실행 수, 관계 수, 워밍업 이후 지연 시간 중앙값을 집계한다.

In [ ]:
from statistics import median


def summarize(report):
    summaries = {}
    for language in ("ko", "en"):
        language_runs = [r for r in report["runs"] if r["language"] == language]
        summaries[language] = {}
        for mode in (*MODELS, "llm_only", "hybrid_base", "hybrid_multi"):
            runs = [r for r in language_runs if r["mode"] == mode]

            def count(index, predicate):
                return sum(any(predicate(e) for e in run["prepared_result"]["entities"])
                           for run in runs if run["message_index"] == index)

            oracle = "오라클" if language == "ko" else "Oracle"
            item = {
                "runs": len(runs), "repeats_per_message": report["repeats"],
                "gcp_exact_organization": count(1, lambda e: e["name"] == "GCP" and e["type"] == "ORGANIZATION"),
                "gcp_with_korean_particle": count(1, lambda e: e["name"] == "GCP에서"),
                "cloud_dns_exact": count(1, lambda e: e["name"] == "Cloud DNS"),
                "oracle_organization": count(4, lambda e: e["name"] == oracle and e["type"] == "ORGANIZATION"),
                "oracle_person": count(4, lambda e: e["name"] == oracle and e["type"] == "PERSON"),
                "oracle_entities": sorted({(e["name"], e["type"]) for r in runs
                    if r["message_index"] == 4 for e in r["prepared_result"]["entities"]}),
                "relation_count": sum(len(r["prepared_result"]["relations"]) for r in runs),
            }
            measured = [r["elapsed_ms"] for r in runs if "elapsed_ms" in r]
            if measured:
                item["warmed_median_ms"] = round(median(measured), 2)
            summaries[language][mode] = item
    return summaries

### 실행 또는 결과 불러오기

In [ ]:
if RUN_EXPERIMENT:
    ontology, configs, extractors, prepare = build_components()
    report = await run_extraction(REPEATS, ontology, configs, extractors, prepare)
    await run_probes(report, extractors, prepare)
    report["summary"] = summarize(report)
    report["overall_latency_ms"] = {
        mode: round(median(r["elapsed_ms"] for r in report["runs"] if r["mode"] == mode), 2)
        for mode in (*MODELS, "llm_only")
    }
    report["sources"] = [
        {"title": "GLiNER2.5 Base official model card", "url": "https://huggingface.co/fastino/gliner2.5-base-v1"},
        {"title": "GLiNER2.5 Multi official model card", "url": "https://huggingface.co/fastino/gliner2.5-multi-v1"},
    ]
    report["status"] = "complete"
    report["completed_at"] = datetime.now(timezone.utc).isoformat()
    save(RESULT_PATH, report)
else:
    report = json.loads(RESULT_PATH.read_text())

print(f"✓ status={report['status']} runs={len(report['runs'])} probes={len(report['controlled_probes'])}")
print(f"  versions={report['versions']}")

## 결과 분석

표는 `report`에서 다시 계산한다. 셀 출력 없이 읽을 수 있도록, 각 표 아래에 2026-11-09 실행의 값과 해석을 함께 기록한다.

In [ ]:
from collections import Counter

from IPython.display import Markdown, display


def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def entity_summary(runs, with_confidence=False):
    """(name, type)별 등장 실행 수와 최대 confidence를 `<br>`로 이어 붙인다."""
    counts, confidence = Counter(), {}
    for run in runs:
        for entity in run["prepared_result"]["entities"]:
            key = (entity["name"], entity["type"])
            confidence[key] = max(confidence.get(key, 0), entity["confidence"])
        counts.update({(e["name"], e["type"]) for e in run["prepared_result"]["entities"]})
    items = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return "<br>".join(
        f"{name}({entity_type}) {count}/{len(runs)}"
        + (f" · {confidence[name, entity_type]:.2f}" if with_confidence else "")
        for (name, entity_type), count in items
    ) or "–"

### 관찰 지표

언어별로 주요 엔티티가 3회 중 몇 번의 실행에서 나타났는지 표시한다. 관계 수는 해당 언어의 12회 실행(문장 4개 × 3회) 합계이다.

In [ ]:
MODES = ["gliner_base", "gliner_multi", "llm_only", "hybrid_base", "hybrid_multi"]
METRICS = {
    "gcp_exact_organization": "GCP(ORG)",
    "gcp_with_korean_particle": "GCP에서",
    "cloud_dns_exact": "Cloud DNS",
    "oracle_organization": "Oracle(ORG)",
    "oracle_person": "Oracle(PERSON)",
}
for language, title in (("ko", "한국어"), ("en", "영어")):
    display(Markdown(f"**{title}**"))
    rows = []
    for mode in MODES:
        item = report["summary"][language][mode]
        rows.append([
            mode,
            *(f"{item[key]}/{item['repeats_per_message']}" for key in METRICS),
            item["relation_count"],
            item.get("warmed_median_ms", "–"),
        ])
    show_table(["모드", *METRICS.values(), "관계 수", "중앙값 ms"], rows)

**한국어**

| 모드 | GCP(ORG) | GCP에서 | Cloud DNS | 오라클(ORG) | 오라클(PERSON) | 관계 수 | 중앙값 ms |
|---|---|---|---|---|---|---|---|
| gliner_base | 0/3 | **3/3** | 0/3 | 0/3 | **3/3** | 0 | 145.86 |
| gliner_multi | 0/3 | **3/3** | 0/3 | 3/3 | 0/3 | 0 | 139.41 |
| llm_only | 3/3 | 0/3 | 3/3 | 3/3 | 0/3 | 5 | 1,248.72 |
| hybrid_base | 3/3 | **3/3** | 3/3 | 3/3 | **3/3** | 5 | – |
| hybrid_multi | 3/3 | **3/3** | 3/3 | 3/3 | 0/3 | 5 | – |

**영어**

| 모드 | GCP(ORG) | Cloud DNS | Oracle(ORG) | Oracle(PERSON) | 관계 수 | 중앙값 ms |
|---|---|---|---|---|---|---|
| gliner_base | 0/3 | 0/3 | 3/3 | 0/3 | 3 | 145.27 |
| gliner_multi | 3/3 | 0/3 | 3/3 | 0/3 | 0 | 148.48 |
| llm_only | 3/3 | 0/3* | 3/3 | 0/3 | 15 | 1,502.76 |
| hybrid_base | 3/3 | 0/3* | 3/3 | 0/3 | 18 | – |
| hybrid_multi | 3/3 | 0/3* | 3/3 | 0/3 | 15 | – |

\* LLM은 영어 메시지 1에서 `Cloud DNS zones`를 3/3 추출했다. 지표가 이름 완전 일치로 계산되어 0으로 집계된다.

**해석**

- 한국어에서 GLiNER 단독은 두 체크포인트 모두 `GCP`를 조사와 함께 추출하고 `Cloud DNS`를 찾지 못한다.
- 영어에서는 PERSON 오류가 사라지고 multi가 `GCP`를 정확히 추출한다. 한국어 입력에서 GLiNER의 품질이 크게 떨어진다.
- 관계는 대부분 LLM이 생성한다. GLiNER 단독 관계는 영어 base의 `Oracle -MENTIONS-> static IP feature` 3개뿐이다.

### 메시지별 엔티티와 confidence

문장·언어·모드별 최종 엔티티이다. `n/3`은 등장 횟수, 마지막 숫자는 최대 confidence이다.

In [ ]:
for pair in report["message_pairs"]:
    display(Markdown(f"**메시지 {pair['index']}** — {pair['ko']}"))
    show_table(
        ["모드", "한국어", "영어"],
        [[mode, *(entity_summary([r for r in report["runs"]
                                  if r["mode"] == mode and r["language"] == language
                                  and r["message_index"] == pair["index"]],
                                 with_confidence=True)
                  for language in ("ko", "en"))]
         for mode in MODES[:3]],
    )

| # | 모드 | 한국어 | 영어 |
|---|---|---|---|
| 1 | gliner_base | GCP에서(ORG) 3/3 · 0.87 | – |
| 1 | gliner_multi | GCP에서(ORG) 3/3 · 0.98 | GCP(ORG) 3/3 · 0.97<br>recurring(EVENT) 3/3 · 0.55 |
| 1 | llm_only | GCP(ORG) 3/3<br>Cloud DNS(OBJECT) 3/3 | GCP(ORG) 3/3<br>Cloud DNS zones(OBJECT) 3/3 |
| 2 | gliner_base | – | screens 0.65 · saving announcements 0.85 · studio information 0.90 (모두 OBJECT 3/3) |
| 2 | gliner_multi | 공지 저장(EVENT) 3/3 · 0.78 | saving announcements(EVENT) 3/3 · studio information(OBJECT) 3/3 |
| 2 | llm_only | 작업실 정보 3/3, 공지 저장 2/3, 서버 쪽 작업 1/3, 공지 1/3 | server-side tasks, screens, announcements, studio information 각 1/3 |
| 3 | gliner_base | – | November usage(EVENT) · **service(ORG) 0.97** · items(OBJECT) 3/3 |
| 3 | gliner_multi | **11월(EVENT) 3/3 · 0.74** | November(EVENT) · **service(ORG) 0.99** · items · charges 3/3 |
| 3 | llm_only | – | November(EVENT), usage, service(OBJECT), items, charges 각 3/3 |
| 4 | gliner_base | **오라클(PERSON) 3/3 · 0.86**<br>IP 기능을(OBJECT) 3/3 · 0.50 | Oracle(ORG) 3/3 · 1.00<br>static IP feature(OBJECT) 3/3 · 0.90 |
| 4 | gliner_multi | 오라클(ORG) 3/3 · 0.98 | Oracle(ORG) 3/3 · 1.00 |
| 4 | llm_only | 오라클(ORG) 3/3 · 0.99 | Oracle(ORG) 3/3 · 0.99<br>static IP feature(OBJECT) 3/3 |

**해석**

- GLiNER는 같은 입력에 3회 모두 같은 결과를 반환한다. LLM은 메시지 2에서 회차마다 다른 엔티티를 반환한다.
- 영어 메시지 2에서는 LLM이 1/3만 찾은 `studio information`을 GLiNER가 3/3 찾는다. GLiNER가 재현율을 보완하는 사례는 영어에서만 관찰된다.
- 반면 `service`를 ORGANIZATION으로 분류한 오류는 confidence 0.97~0.99로 매우 높다. 오류의 confidence가 정답과 구분되지 않는다는 점은 한국어 결과와 같다.

### 통제 프로브 결과

In [ ]:
probes = {}
for probe in report["controlled_probes"]:
    probes.setdefault(probe["probe"], {"text": probe["text"]})[probe["mode"]] = ", ".join(
        f"{e['name']}({e['type']})" for e in probe["prepared_result"]["entities"]) or "–"
show_table(
    ["프로브", "입력", "gliner_base", "gliner_multi"],
    [[label, item["text"], item["gliner_base"], item["gliner_multi"]] for label, item in probes.items()],
)

| 프로브 | gliner_base | gliner_multi |
|---|---|---|
| 한국어 문맥 + `Oracle` | Oracle(ORG), IP 기능을(OBJECT) | Oracle(ORG) |
| 영어 문맥 + `오라클` | 오라클(ORG), static IP feature(OBJECT) | **오라클(PERSON)** |
| `GCP 에서` (조사 띄어쓰기) | **GCP(ORG)** | **GCP(ORG)** |

**해석**

- **조사**: 조사를 띄어 쓰면 두 체크포인트 모두 정확한 범위의 `GCP`를 추출한다. 범위 오류는 GLiNER가 공백 단위 토큰을 엔티티 경계로 사용하여 교착어의 조사를 분리하지 못하기 때문이다.
- **오라클 타입**: base는 한국어 문맥의 한글 `오라클`에서만 PERSON으로 오분류하고, 표기나 문맥 중 하나만 바꾸면 ORGANIZATION으로 분류한다. multi는 반대로 영어 문맥의 한글 `오라클`을 PERSON으로 분류한다. 두 체크포인트 모두 문자 체계가 섞인 입력에서 타입 판단이 불안정하다.
- 한국어 대화에는 영어 고유명사의 한글 표기와 영문 표기가 함께 나타나므로, 체크포인트 교체만으로는 이 불안정성을 피할 수 없다.

### 지연 시간

In [ ]:
show_table(
    ["모드", "워밍업 ms", "전체 중앙값 ms", "한국어 중앙값 ms", "영어 중앙값 ms"],
    [[mode,
      round(next(w["elapsed_ms"] for w in report["warmup"] if w["mode"] == mode)),
      report["overall_latency_ms"][mode],
      report["summary"]["ko"][mode]["warmed_median_ms"],
      report["summary"]["en"][mode]["warmed_median_ms"]]
     for mode in report["overall_latency_ms"]],
)

| 모드 | 워밍업 ms | 전체 중앙값 ms | 한국어 | 영어 |
|---|---|---|---|---|
| gliner_base | 6,365 | 145.86 | 145.86 | 145.27 |
| gliner_multi | 7,357 | 147.67 | 139.41 | 148.48 |
| llm_only | 2,263 | 1,404.26 | 1,248.72 | 1,502.76 |

**해석**

워밍업 이후 GLiNER는 CPU에서 문장당 약 0.15초로, 네트워크 왕복을 포함한 LLM보다 약 10배 빠르다. 두 체크포인트의 속도 차이는 없다.
다만 Hybrid 구성은 LLM 호출을 포함하므로 전체 지연 시간은 LLM이 결정한다. GLiNER의 속도 이점은 GLiNER 단독 구성에서만 의미가 있으며, 한국어에서는 단독 구성의 품질이 충분하지 않다.

## 결론과 한계

**결론**

1. 한국어 입력에서 GLiNER 단독 추출은 조사 포함 범위, 타입 오류, `Cloud DNS` 누락으로 사용하기 어렵다.
2. multi 체크포인트는 한국어 메시지 4의 PERSON 오류를 해소하지만, 조사 문제는 그대로이고 영어 문맥의 한글 표기에서 새로운 PERSON 오류를 보인다.
3. 한국어에서 Hybrid는 LLM 결과에 GLiNER 오류를 더할 뿐 정답을 보완하지 않았다. 영어에서는 일부 재현율 보완이 있었으나 고신뢰 오분류도 함께 추가되었다.
4. 현재 한국어 대화 메모리에는 LLM-only 구성이 가장 깨끗한 결과를 낸다. GLiNER를 유지하려면 조사 분리 등 형태소 단위 전처리를 먼저 검토해야 한다.

**한계**

- 문장 쌍 4개 × 3회의 진단용 표본이며 정답 레이블이 없다. 일반적인 언어별 성능 비교가 아니다.
- 영어 문장은 직접 번역한 것으로, 번역 과정에서 표현 차이(예: `Cloud DNS zones`)가 생긴다.
- 지표가 이름 완전 일치로 계산되므로 의미상 정답인 변형이 누락으로 집계될 수 있다.
- 지연 시간은 단일 머신의 CPU 실행 결과이며, LLM 시간은 네트워크 상태에 따라 달라진다.